In [3]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# A session that automatically retries when ChEMBL's server has a hiccup (error 500 etc.),
# waiting a little longer before each new attempt
session = requests.Session()
retries = Retry(total=8, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))


def get_activities(target_id):
    url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
    params = {"target_chembl_id": target_id, "pchembl_value__isnull": "false", "limit": 200}
    rows = []
    while url:
        r = session.get(url, params=params, timeout=90)
        r.raise_for_status()
        page = r.json()
        rows += page["activities"]
        # ChEMBL sends results in pages; follow the link to the next page
        nxt = page["page_meta"]["next"]
        url = "https://www.ebi.ac.uk" + nxt if nxt else None
        params = None  # the next-page link already includes the filters
        print(f"Downloaded {len(rows)} of {page['page_meta']['total_count']} records")
    return pd.DataFrame(rows)


df = get_activities("CHEMBL3714531")
df.to_csv("../data/notum_raw.csv", index=False)

Downloaded 200 of 783 records
Downloaded 400 of 783 records
Downloaded 600 of 783 records
Downloaded 783 of 783 records


OSError: Cannot save file into a non-existent directory: '..\data'